In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, logistic, gumbel_r

pd.options.display.float_format = "{:.5f}".format

In [2]:
deciduous_teeth = pd.read_csv("../data/deciduous_teeth.csv", sep=";")

In [3]:
permanent_teeth = pd.read_csv("../data/permanent_teeth.csv", sep=";")

In [4]:
def preprocess_data_lifelines(df):
    df = df.copy()
    df.loc[((df["LOWER"].isna()), "LOWER")] = 0
    df.loc[((df["UPPER"].isna()), "UPPER")] = np.inf

    return df


deciduous_teeth = preprocess_data_lifelines(deciduous_teeth)
permanent_teeth = preprocess_data_lifelines(permanent_teeth)

In [6]:
deciduous_preds_ll = pd.read_csv(f"preds\\ll_tooth_preds_dec.csv", sep=";")
deciduous_preds_xgb = pd.read_csv(f"preds\\xgb_tooth_preds_dec.csv", sep=";")
deciduous_preds_cb = pd.read_csv(f"preds\\cb_tooth_preds_dec.csv", sep=";")

In [7]:
# filter ll preds to have same order as deciduous_teeth
deciduous_preds_ll["TOOTH_NUMBER"] = pd.Categorical(
    deciduous_preds_ll["TOOTH_NUMBER"], deciduous_teeth["TOOTH_NUMBER"].unique()
)
deciduous_preds_ll = deciduous_preds_ll.sort_values(
    ["DOG", "TOOTH_NUMBER", "DEVELOPM_STAGE"], ignore_index=True
)

In [8]:
# add dog, tooth_nr, developm_stage to xgb and cb
# before no info dogs get filtered out!
deciduous_preds_xgb["DOG"] = deciduous_teeth["DOG"]
deciduous_preds_xgb["TOOTH_NUMBER"] = deciduous_teeth["TOOTH_NUMBER"]
deciduous_preds_xgb["DEVELOPM_STAGE"] = deciduous_teeth["DEVELOPM_STAGE"]
deciduous_preds_cb["DOG"] = deciduous_teeth["DOG"]
deciduous_preds_cb["TOOTH_NUMBER"] = deciduous_teeth["TOOTH_NUMBER"]
deciduous_preds_cb["DEVELOPM_STAGE"] = deciduous_teeth["DEVELOPM_STAGE"]

In [9]:
permanent_preds_ll = pd.read_csv("./preds/ll_tooth_preds_perma.csv", sep=";")
permanent_preds_xgb = pd.read_csv("./preds/xgb_tooth_preds_perma.csv", sep=";")
permanent_preds_cb = pd.read_csv("./preds/cb_tooth_preds_perma.csv", sep=";")

In [10]:
# filter ll preds to have same order as permanent
permanent_preds_ll["TOOTH_NUMBER"] = pd.Categorical(
    permanent_preds_ll["TOOTH_NUMBER"], permanent_teeth["TOOTH_NUMBER"].unique()
)
permanent_preds_ll = permanent_preds_ll.sort_values(
    ["DOG", "TOOTH_NUMBER", "DEVELOPM_STAGE"], ignore_index=True
)

In [11]:
# add dog, tooth_nr, developm_stage to xgb and cb
# before no info dogs get filtered out!
permanent_preds_xgb["DOG"] = permanent_teeth["DOG"]
permanent_preds_xgb["TOOTH_NUMBER"] = permanent_teeth["TOOTH_NUMBER"]
permanent_preds_xgb["DEVELOPM_STAGE"] = permanent_teeth["DEVELOPM_STAGE"]
permanent_preds_cb["DOG"] = permanent_teeth["DOG"]
permanent_preds_cb["TOOTH_NUMBER"] = permanent_teeth["TOOTH_NUMBER"]
permanent_preds_cb["DEVELOPM_STAGE"] = permanent_teeth["DEVELOPM_STAGE"]

In [12]:
def get_distribution(dist_name):
    if dist_name == "Normal" or dist_name == "normal":
        return norm
    elif dist_name == "Logistic" or dist_name == "logistic":
        return logistic
    elif dist_name == "Extreme" or dist_name == "extreme":
        return gumbel_r
    else:
        raise ValueError("Invalid distribution name")


def compute_survival_probability(preds, timepoint, dist_name="Normal", scale=1.0):
    dist = get_distribution(dist_name)
    log_time = np.log(timepoint)
    survival_probs = 1 - dist.cdf((log_time - preds) / scale)
    return survival_probs

In [25]:
# df moet LOWER unknown als 0 hebben en UPPER unknown als inf
# obv: https://www.tidymodels.org/learn/statistics/survival-metrics-details/#converting-censored-data-to-binary-data
def get_teeth_of_dog(df, dog_id, t, stage):
    # select dog
    dog = df[df["DOG"] == dog_id]
    # only select ONE stage!!!
    dog = dog[dog["DEVELOPM_STAGE"] == stage]
    # maak nieuwe kolom OBSERVED_t die bijhoudt of survived is op tijdstip t
    dog[f"ERRUPTED"] = np.nan

    # geen censoring: als t < LOWER/UPPER dan survived anders niet
    dog.loc[(dog["CENS"] == 0) & (t >= dog["LOWER"]), "ERRUPTED"] = 1
    dog.loc[(dog["CENS"] == 0) & (t < dog["LOWER"]), "ERRUPTED"] = 0

    # left censoring: als t >= UPPER dan happened anders unknown
    dog.loc[(dog["CENS"] == 2) & (t >= dog["UPPER"]), "ERRUPTED"] = 1

    # interval censoring: als t < LOWER dan survived, als t >= UPPER dan event en als t in [LOWER, UPPER] dan unknown
    dog.loc[(dog["CENS"] == 3) & (t >= dog["UPPER"]), "ERRUPTED"] = 1
    dog.loc[(dog["CENS"] == 3) & (t <= dog["LOWER"]), "ERRUPTED"] = 0

    # right censoring: als t < LOWER dan survived anders unknown
    dog.loc[(dog["CENS"].isin([4, 5])) & (t <= dog["LOWER"]), "ERRUPTED"] = 0

    dog["check_up_time"] = t
    dog["DOG"] = dog_id

    return dog

In [ ]:
def get_dogs_s_preds(dog_teeth, timepoint, preds_ll, preds_xgb, preds_cb):
    # voor alle drie de modellen
    preds = pd.DataFrame()
    preds["ERRUPTED"] = dog_teeth["ERRUPTED"]

    filtered_ll = preds_ll[
        preds_ll.set_index(["TOOTH_NUMBER", "DEVELOPM_STAGE"]).index.isin(
            dog_teeth.set_index(["TOOTH_NUMBER", "DEVELOPM_STAGE"]).index
        )
    ]
    preds["LL"] = list(
        filtered_ll.loc[filtered_ll["time"] == timepoint, "pred_survival"].reset_index(
            drop=True
        )
    )

    # no loop needed bc dist & scale same for single dog!
    # need to get tooth / stage combo --> to know what rows to get from preds --> compute survival
    filtered_xgb = preds_xgb[
        preds_xgb.set_index(["TOOTH_NUMBER", "DEVELOPM_STAGE"]).index.isin(
            dog_teeth.set_index(["TOOTH_NUMBER", "DEVELOPM_STAGE"]).index
        )
    ]
    preds["XGB"] = compute_survival_probability(
        filtered_xgb["pred"].values,
        timepoint,
        filtered_xgb["dist"].min(),
        filtered_xgb["scale"].min(),
    )

    filtered_cb = preds_cb[
        preds_cb.set_index(["TOOTH_NUMBER", "DEVELOPM_STAGE"]).index.isin(
            dog_teeth.set_index(["TOOTH_NUMBER", "DEVELOPM_STAGE"]).index
        )
    ]
    preds["CB"] = compute_survival_probability(
        filtered_cb["pred"].values,
        timepoint,
        filtered_cb["dist"].min(),
        filtered_cb["scale"].min(),
    )

    preds["DOG"] = dog_teeth["DOG"]
    preds["check_up_time"] = dog_teeth["check_up_time"]

    return preds

In [ ]:
def get_dog_age_ll(
    df, dog, timepoint, timepoint_check_up, preds_ll, preds_xgb, preds_cb, stage
):
    dog_teeth = get_teeth_of_dog(df, dog, timepoint_check_up, stage)
    # leave out unknowns --> meaning multiply by 1
    dog_teeth = dog_teeth[dog_teeth["ERRUPTED"].notna()]

    preds = get_dogs_s_preds(dog_teeth, timepoint, preds_ll, preds_xgb, preds_cb)

    eps = 1e-12
    erupted = preds["ERRUPTED"] == 1

    ll_ll = np.sum(
        np.log(np.clip(np.where(erupted, 1 - preds["LL"], preds["LL"]), eps, 1))
    )
    ll_xgb = np.sum(
        np.log(np.clip(np.where(erupted, 1 - preds["XGB"], preds["XGB"]), eps, 1))
    )
    ll_cb = np.sum(
        np.log(np.clip(np.where(erupted, 1 - preds["CB"], preds["CB"]), eps, 1))
    )

    return (ll_ll, ll_xgb, ll_cb)

In [34]:
TIMES_d = []
TIMES_d.extend(deciduous_teeth["LOWER"].unique())
TIMES_d.extend(deciduous_teeth["UPPER"].unique())
TIMES_d = set(TIMES_d)
TIMES_d.remove(np.inf)
TIMES_d.remove(0)
TIMES_d = list(TIMES_d)

In [35]:
TIMES_p = []
TIMES_p.extend(permanent_teeth["LOWER"].unique())
TIMES_p.extend(permanent_teeth["UPPER"].unique())
TIMES_p = set(TIMES_p)
TIMES_p.remove(np.inf)
TIMES_p.remove(0)
TIMES_p = list(TIMES_p)

In [ ]:
def get_ll_lists(df, age_threshold, df_ll, df_xgb, df_cb, stage, TIMES):
    # voor iedere hond op alle tijdstippen LLs berekenen voor iedere momentopname vd hond
    # get ground truth 
    lls_ll_test = []
    lls_xgb_test = []
    lls_cb_test = []

    for dog in df["DOG"].unique():
        filtered_ll = df_ll[df_ll["DOG"] == dog]
        filtered_xgb = df_xgb[df_xgb["DOG"] == dog]
        filtered_cb = df_cb[df_cb["DOG"] == dog]

        check_up_times = []
        check_up_times.extend(df[df["DOG"] == dog]["LOWER"].unique())
        check_up_times.extend(df[df["DOG"] == dog]["UPPER"].unique())
        check_up_times = set(check_up_times)
        if np.inf in check_up_times:
            check_up_times.remove(np.inf)
        if 0 in check_up_times:
            check_up_times.remove(0)
        check_up_times = list(check_up_times)
        check_up_times.sort()

        for check_up_time in check_up_times:
            ground_truth = check_up_time >= age_threshold
            print(f"calc for dog {dog} w/ checkup time {check_up_time} started")
            for time in TIMES:
                try:
                    lls = get_dog_age_ll(
                        df,
                        dog,
                        time,
                        check_up_time,
                        filtered_ll,
                        filtered_xgb,
                        filtered_cb,
                        stage,
                    )
                    lls_ll_test.append(
                        {
                            "dog": dog,
                            "checkup": check_up_time,
                            "time": time,
                            "ground_truth": ground_truth,
                            "ll": lls[0],
                        }
                    )
                    lls_xgb_test.append(
                        {
                            "dog": dog,
                            "checkup": check_up_time,
                            "time": time,
                            "ground_truth": ground_truth,
                            "ll": lls[1],
                        }
                    )
                    lls_cb_test.append(
                        {
                            "dog": dog,
                            "checkup": check_up_time,
                            "time": time,
                            "ground_truth": ground_truth,
                            "ll": lls[2],
                        }
                    )
                except Exception as e:
                    print(f"error {e} in {dog} at time {check_up_time}")
                    lls_ll_test.append(
                        {
                            "dog": dog,
                            "checkup": check_up_time,
                            "time": time,
                            "ground_truth": ground_truth,
                            "ll": np.nan,
                        }
                    )
                    lls_xgb_test.append(
                        {
                            "dog": dog,
                            "checkup": check_up_time,
                            "time": time,
                            "ground_truth": ground_truth,
                            "ll": np.nan,
                        }
                    )
                    lls_cb_test.append(
                        {
                            "dog": dog,
                            "checkup": check_up_time,
                            "time": time,
                            "ground_truth": ground_truth,
                            "ll": np.nan,
                        }
                    )
    return lls_ll_test, lls_xgb_test, lls_cb_test

## Deciduous

In [ ]:
lls_ll_test, lls_xgb_test, lls_cb_test = get_ll_lists(
    deciduous_teeth,
    56,
    deciduous_preds_ll,
    deciduous_preds_xgb,
    deciduous_preds_cb,
    1,
    TIMES_d,
)

lls_ll_df = pd.DataFrame(lls_ll_test)
lls_xgb_df = pd.DataFrame(lls_xgb_test)
lls_cb_df = pd.DataFrame(lls_cb_test)

lls_ll_df.to_csv("./dental_likelihoods/dental_likelihoods_ll_dec_stage1.csv", sep=";")
lls_xgb_df.to_csv("./dental_likelihoods/dental_likelihoods_xgb_dec_stage1.csv", sep=";")
lls_cb_df.to_csv("./dental_likelihoods/dental_likelihoods_cb_dec_stage1.csv", sep=";")

In [ ]:
lls_ll_test, lls_xgb_test, lls_cb_test = get_ll_lists(
    deciduous_teeth,
    56,
    deciduous_preds_ll,
    deciduous_preds_xgb,
    deciduous_preds_cb,
    2,
    TIMES_d,
)

lls_ll_df = pd.DataFrame(lls_ll_test)
lls_xgb_df = pd.DataFrame(lls_xgb_test)
lls_cb_df = pd.DataFrame(lls_cb_test)

lls_ll_df.to_csv("./dental_likelihoods/dental_likelihoods_ll_dec_stage2.csv", sep=";")
lls_xgb_df.to_csv("./dental_likelihoods/dental_likelihoods_xgb_dec_stage2.csv", sep=";")
lls_cb_df.to_csv("./dental_likelihoods/dental_likelihoods_cb_dec_stage2.csv", sep=";")

## Permanent

In [ ]:
lls_ll_test, lls_xgb_test, lls_cb_test = get_ll_lists(
    permanent_teeth,
    105,
    permanent_preds_ll,
    permanent_preds_xgb,
    permanent_preds_cb,
    1,
    TIMES_p,
)

lls_ll_df = pd.DataFrame(lls_ll_test)
lls_xgb_df = pd.DataFrame(lls_xgb_test)
lls_cb_df = pd.DataFrame(lls_cb_test)

lls_ll_df.to_csv("./dental_likelihoods/dental_likelihoods_ll_perma_stage1.csv", sep=";")
lls_xgb_df.to_csv("./dental_likelihoods/dental_likelihoods_xgb_perma_stage1.csv", sep=";")
lls_cb_df.to_csv("./dental_likelihoods/dental_likelihoods_cb_perma_stage1.csv", sep=";")

In [ ]:
lls_ll_test, lls_xgb_test, lls_cb_test = get_ll_lists(
    permanent_teeth,
    105,
    permanent_preds_ll,
    permanent_preds_xgb,
    permanent_preds_cb,
    2,
    TIMES_p,
)

lls_ll_df = pd.DataFrame(lls_ll_test)
lls_xgb_df = pd.DataFrame(lls_xgb_test)
lls_cb_df = pd.DataFrame(lls_cb_test)

lls_ll_df.to_csv("./dental_likelihoods/dental_likelihoods_ll_perma_stage2.csv", sep=";")
lls_xgb_df.to_csv("./dental_likelihoods/dental_likelihoods_xgb_perma_stage2.csv", sep=";")
lls_cb_df.to_csv("./dental_likelihoods/dental_likelihoods_cb_perma_stage2.csv", sep=";")

In [ ]:
lls_ll_test, lls_xgb_test, lls_cb_test = get_ll_lists(
    permanent_teeth,
    105,
    permanent_preds_ll,
    permanent_preds_xgb,
    permanent_preds_cb,
    3,
    TIMES_p,
)

lls_ll_df = pd.DataFrame(lls_ll_test)
lls_xgb_df = pd.DataFrame(lls_xgb_test)
lls_cb_df = pd.DataFrame(lls_cb_test)

lls_ll_df.to_csv("./dental_likelihoods/dental_likelihoods_ll_perma_stage3.csv", sep=";")
lls_xgb_df.to_csv("./dental_likelihoods/dental_likelihoods_xgb_perma_stage3.csv", sep=";")
lls_cb_df.to_csv("./dental_likelihoods/dental_likelihoods_cb_perma_stage3.csv", sep=";")

calc for dog 1 w/ checkup time 91.0 started
calc for dog 1 w/ checkup time 93.0 started
calc for dog 1 w/ checkup time 98.0 started
calc for dog 1 w/ checkup time 99.0 started
calc for dog 1 w/ checkup time 100.0 started
calc for dog 1 w/ checkup time 101.0 started
calc for dog 1 w/ checkup time 105.0 started
calc for dog 1 w/ checkup time 107.0 started
calc for dog 1 w/ checkup time 114.0 started
calc for dog 1 w/ checkup time 117.0 started
calc for dog 1 w/ checkup time 119.0 started
calc for dog 1 w/ checkup time 121.0 started
calc for dog 1 w/ checkup time 122.0 started
calc for dog 1 w/ checkup time 124.0 started
calc for dog 1 w/ checkup time 128.0 started
calc for dog 1 w/ checkup time 129.0 started
calc for dog 1 w/ checkup time 131.0 started
calc for dog 1 w/ checkup time 132.0 started
calc for dog 1 w/ checkup time 133.0 started
calc for dog 1 w/ checkup time 135.0 started
calc for dog 1 w/ checkup time 136.0 started
calc for dog 2 w/ checkup time 110.0 started
calc for dog 2

# Get survival probabilities

In [ ]:
preds_st_list = []

deciduous_teeth["ERRUPTED"] = 1

for dog in deciduous_teeth["DOG"].unique():
    check_up_times = []
    check_up_times.extend(
        deciduous_teeth[deciduous_teeth["DOG"] == dog]["LOWER"].unique()
    )
    check_up_times.extend(
        deciduous_teeth[deciduous_teeth["DOG"] == dog]["UPPER"].unique()
    )
    check_up_times = set(check_up_times)
    if np.inf in check_up_times:
        check_up_times.remove(np.inf)
    if 0 in check_up_times:
        check_up_times.remove(0)
    check_up_times = list(check_up_times)
    check_up_times.sort()

    for check_up_time in check_up_times:
        dog_teeth = deciduous_teeth[deciduous_teeth["DOG"] == dog]
        dog_teeth["check_up_time"] = check_up_time

        filtered_ll = deciduous_preds_ll[deciduous_preds_ll["DOG"] == dog]
        filtered_xgb = deciduous_preds_xgb[deciduous_preds_xgb["DOG"] == dog]
        filtered_cb = deciduous_preds_cb[deciduous_preds_cb["DOG"] == dog]

        # get surv for all TEETH!
        preds_st_list.append(
            get_dogs_s_preds(dog_teeth, 56, filtered_ll, filtered_xgb, filtered_cb)
        )

st_df = pd.concat(preds_st_list)
st_df.to_csv("survival_probs_in/st_df_dec.csv", sep=";")

In [ ]:
preds_st_list_perma = []
permanent_teeth["ERRUPTED"] = 1

for dog in permanent_teeth["DOG"].unique():
    check_up_times = []
    check_up_times.extend(
        permanent_teeth[permanent_teeth["DOG"] == dog]["LOWER"].unique()
    )
    check_up_times.extend(
        permanent_teeth[permanent_teeth["DOG"] == dog]["UPPER"].unique()
    )
    check_up_times = set(check_up_times)
    if np.inf in check_up_times:
        check_up_times.remove(np.inf)
    if 0 in check_up_times:
        check_up_times.remove(0)
    check_up_times = list(check_up_times)
    check_up_times.sort()

    for check_up_time in check_up_times:
        dog_teeth = permanent_teeth[permanent_teeth["DOG"] == dog]
        dog_teeth["check_up_time"] = check_up_time

        filtered_ll = permanent_preds_ll[permanent_preds_ll["DOG"] == dog]
        filtered_xgb = permanent_preds_xgb[permanent_preds_xgb["DOG"] == dog]
        filtered_cb = permanent_preds_cb[permanent_preds_cb["DOG"] == dog]

        preds_st_list_perma.append(
            get_dogs_s_preds(dog_teeth, 105, filtered_ll, filtered_xgb, filtered_cb)
        )

st_df_perma = pd.concat(preds_st_list_perma)
st_df_perma.to_csv("survival_probs_in/st_df_perma.csv", sep=";")